# Module 39.3 — Fine-Tuning
---

## 1. What is Fine-Tuning?

Fine-tuning is a transfer learning technique in which we take a pretrained neural network and continue training some or all of its existing layers using a new target dataset.

Unlike feature extraction, where the backbone remains frozen, fine-tuning allows selected pretrained layers to update their weights.

Let's continue our industrial motor defect classification example.

Our target classes are:

- Normal
- Crack
- Surface defect
- Deformation

We already have a pretrained ResNet18 model that learned visual representations from ImageNet.

Instead of keeping the entire backbone fixed, we allow selected layers to adapt to industrial defect images.

### Fine-tuning workflow

```text
Pretrained Backbone
        ↓
Unfreeze Selected Layers
        ↓
Small Learning Rate
        ↓
Task-Specific Training
        ↓
Adapted Model
```

![Domain adaptation and transfer learning across multiple datasets](https://images.openai.com/static-rsc-4/ZxgdQ7oQoFiCmwmhtj7Z8WliFqFjNBRMW-_G78Lja-xWpuhFG9dhg497jfWh_dDBMtTp2DI0qpaUNZX7QOO1kL04PIfXVcfGXzlcTtW25Yi69kQiR3RkUdA9hCMwQFZqurlzsUxRFMzOi60_K78R9JReo1bG60630qe-CTPV_M0?purpose=inline)

[Source: Frontiers — Domain Adaptation and Transfer Learning](https://www.frontiersin.org/journals/computer-science/articles/10.3389/fcomp.2025.1639421/full)

**Central idea:**

> Fine-tuning preserves useful pretrained knowledge while adapting the model's existing representations to a new problem.

---

## 2. Why Do We Need Fine-Tuning?

Remember our previous topic, feature extraction.

In feature extraction:

- The pretrained backbone is frozen.
- Only the new classification head is trained.
- The backbone cannot adapt its learned features.

But consider this situation:

A ResNet18 pretrained on ImageNet has learned to recognise general visual patterns from natural photographs.

Now we want it to recognise:

- Tiny surface cracks.
- Metal deformation.
- Winding insulation damage.
- Manufacturing defects.

The pretrained representations may not be sufficiently specialised for these images.

Fine-tuning allows the backbone to learn additional task-specific features.

![ImageNet classification using deep convolutional neural networks](https://images.openai.com/static-rsc-4/aYj9VDZne2-dP7LaFecS58ICxWtkV6Ar_Bbo_wfvh9nEIElx6lJStLNve2je3vDfZk-FbtSIxAcetbvwo1av_INiC6UDuR7-VVCJG9WTjbwxa9OtdGjexeD56jqOVxSpo_I5pOpP0jTZL7meLi55Itro6efmfVHtrpZgtmMbtQM?purpose=inline)

[Source: Medium — ImageNet Classification with Deep Convolutional Neural Networks](https://medium.com/)

![Three-phase stator winding failure](https://images.openai.com/static-rsc-4/E-AWNy9IH9K5T3y_ZZ0JRZRpFXiBAj4ZF-vhf9eGoaJ3bGQVF3wjSr5rW4vayBhFhg3a8hUYFJ2rMrQ4K3Zu1ZgTmjEQWdKj75uBKJay8OjdCd8QXunIYDEV_BnlG39Zz1Of37x9jM7RLb466CfIpt3Tj-My5wz57RmY9PIqu0U?purpose=inline)

[Source: Duke Electric — Three-Phase Stator Winding Failure](https://www.dukeelectric.com/)

![Motor insulation breakdown](https://images.openai.com/static-rsc-4/gEGYOds4l4J-GSRUNEqmMaFmCDxRMd3oQqEu81hiDWzINEsIF5O6cAIq-5JVm1bTioFkX0a1lekn2qswv81yubma_SmoeO5HT-nKc3s4u31ranqTc1McZ5nW6ZYsGo3HNa4tqVCAe2Z8s0WNCzb8W9Duh8DZfttT2goTpM5wuHA?purpose=inline)

[Source: Motor Insulation Breakdown](https://www.google.com/search?q=motor+insulation+breakdown)

### Feature extraction vs. fine-tuning

| Feature extraction | Fine-tuning |
|---|---|
| Backbone frozen | Selected backbone layers unfrozen |
| Only classifier learns | Classifier and selected backbone layers learn |
| Existing features remain unchanged | Existing features can adapt |
| Usually faster | Usually more computationally expensive |

---

## 3. Understanding the Three Fine-Tuning Steps

### 3.1 Unfreeze Selected Layers

Unfreezing means allowing a previously frozen layer's weights to be updated during backpropagation.

Remember the hierarchical feature learning concept from CNNs:

![Fine-tuning and transfer learning](https://images.openai.com/static-rsc-4/FxJzg1nJveOwcSylLFbjhJXT821nht5Utonc-iyLUQ2iq4E5xx5nrRsnz3fKJuCTIIP10yPnMkBWgdfu48d4q6hHPOCsDkEzzfij4McXZ0WcshY4gAI-WBPkTNXIaJCMbcgjqg9qu2E6E_KW0DGt-9ycWtZwId6u7rk-zaSMtig?purpose=inline)

[Source: Deep Learning — Fine-Tuning](https://perso.esiee.fr/~chierchg/deep-learning/tutorials/transfer/transfer-3.html)

- Early layers generally learn more generic visual features, such as edges and simple textures.
- Deeper layers generally learn more specialised features.

Therefore, a common strategy is to keep early layers frozen and unfreeze deeper layers.

Example using ResNet18:

| Layer | Fine-tuning status |
|---|---|
| `conv1` and `bn1` | Frozen |
| `layer1` | Frozen |
| `layer2` | Frozen |
| `layer3` | Frozen |
| `layer4` | Trainable |
| `fc` | Trainable |

**Why unfreeze `layer4`?**

It is close to the classification head and contains relatively high-level feature representations that can adapt to the target task.

**Important:** This is a common starting strategy, not a universal rule. The number of layers to unfreeze depends on the target dataset, similarity to pretraining data, and available training examples.

### 3.2 Use a Small Learning Rate

This is one of the most important aspects of fine-tuning.

Remember that pretrained weights already contain useful knowledge.

If we use a very large learning rate, weight updates may become too large and destroy useful pretrained representations.

This problem is associated with **catastrophic forgetting**.

![Deep learning architecture and pretrained model learning](https://images.openai.com/static-rsc-4/Q0Wo8Q38pG8kdShR_ikwpvCirGWIiTP5Bv9JC0Mv3uR5se1YiA2N_hmxwtbjoQBoMtWfKvY4q1M_Ai4FQqPqPrUjYPxo3-mQLAOqGYpw3PHq6FRzTYzxJDHNXuBqHaoAbdGI9kDC0F4Hb8bpJzvkaYC53JSlLCBII1NiT65S9eQ?purpose=inline)

[Source: CS231n Lecture 7 — AFAIK](https://yxxshin.github.io/2022/11/15/2022-11-15-CS231n-Lecture7/)

Consider these illustrative learning rates:

| Component | Learning rate |
|---|---:|
| Pretrained backbone | 0.0001 |
| New classification head | 0.001 |

**Why different learning rates?**

- The pretrained backbone already has useful weights, so it needs smaller adjustments.
- The newly created classification head starts with randomly initialised weights, so it can benefit from a larger learning rate.

This technique is called **discriminative learning rates**.

A smaller learning rate does not guarantee better performance, but it gives the pretrained weights a more controlled opportunity to adapt.

### 3.3 Task-Specific Training

After unfreezing selected layers and setting learning rates, we train the model using our target dataset.

For our motor defect problem:

```text
Input: Motor component image
             ↓
Pretrained ResNet18
             ↓
Adapted deeper CNN layers
             ↓
New classification head
             ↓
Normal / Crack / Surface defect / Deformation
```

During training:

1. The image passes through the network.
2. The model generates class logits.
3. Cross-entropy loss compares predictions with the actual labels.
4. Backpropagation calculates gradients for trainable layers.
5. The optimizer updates the selected backbone layers and classifier.

The frozen layers retain their original weights.

---

## 4. Practical PyTorch Implementation — ResNet18 Fine-Tuning

Let's continue with the ResNet18 model from the previous feature extraction topic.

### Step 1 — Load pretrained ResNet18

```python
import torch
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

print("Device:", device)
```

### Step 2 — Replace the classification head

```python
num_classes = 4

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)
```

Our model now predicts four motor defect categories.

### Step 3 — Freeze all layers

```python
for param in model.parameters():
    param.requires_grad = False
```

Initially, all pretrained parameters are frozen.

### Step 4 — Unfreeze selected layers

```python
for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True
```

Now:

- `layer1`, `layer2`, `layer3` remain frozen.
- `layer4` becomes trainable.
- `fc` becomes trainable.

### Step 5 — Configure different learning rates

```python
model = model.to(device)

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.layer4.parameters(),
            "lr": 0.0001
        },
        {
            "params": model.fc.parameters(),
            "lr": 0.001
        }
    ],
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()
```

Notice that the classification head has a higher learning rate than the pretrained backbone.

### Step 6 — Train the model

Assuming `train_loader` is already prepared:

```python
epochs = 10

for epoch in range(epochs):

    model.train()

    # Keep frozen BatchNorm layers in evaluation mode
    model.bn1.eval()
    model.layer1.eval()
    model.layer2.eval()
    model.layer3.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )
```

**Why keep frozen BatchNorm layers in evaluation mode?**

`requires_grad=False` prevents parameter gradients, but BatchNorm can still update its running mean and variance when the model is in training mode.

Keeping the frozen backbone's BatchNorm layers in evaluation mode prevents those statistics from changing.

The unfrozen `layer4` and classification head remain trainable.

### Step 7 — Validation

After each training epoch, evaluate the model using validation data.

```python
model.eval()

val_correct = 0
val_total = 0

with torch.inference_mode():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        val_correct += (
            (predictions == labels).sum().item()
        )

        val_total += labels.size(0)

val_accuracy = 100 * val_correct / val_total

print(
    f"Validation Accuracy: {val_accuracy:.2f}%"
)
```

Use validation loss, accuracy, and per-class metrics to determine whether fine-tuning actually improves the model.

**Implementation note:** In a complete script, move the model to `device` before constructing the optimizer. Also, save the best validation checkpoint rather than automatically keeping the final epoch.

---

## 5. Feature Extraction vs. Fine-Tuning — Final Comparison

![CNN applications in healthcare diagnostics](https://images.openai.com/static-rsc-4/ZRvFW3Afwzi8ulyZ0xC8yTpixRJeiPoAniVWfDB5TkHxHOnUXG8zRIOVJbur6Db2v6VtSZ3iy6p80V615F19nb1lh82T1EUe7x8GIvA0gqnKMewLiKIa4CcjGOqn3DfyVmcaEMZfEYOccyH6OhQsK94uyA5go2AEJQWW1VKEr0Y?purpose=inline)

[Source: Next Electronics — Healthcare Diagnostics with CNNs](https://next.gr/ai/image-classification/healthcare-diagnostics-with-cnns)

| Property | Feature extraction | Fine-tuning |
|---|---|---|
| Pretrained backbone | Frozen | Selected layers unfrozen |
| Classifier | Trainable | Trainable |
| Backbone weights | Unchanged | Updated selectively |
| Learning rate | Usually higher for new head | Usually lower for pretrained layers |
| Training time | Lower | Higher |
| Adaptability | Limited | Greater |
| Risk of overfitting | Still possible | Can increase with small datasets |

### Recommended workflow

For a new computer vision project, follow this sequence:

**1. Load pretrained model**

ResNet18 / EfficientNet / MobileNet

**2. Feature extraction**

Freeze backbone, train new classifier.

**3. Evaluate validation performance**

**4. Fine-tune selected layers**

Unfreeze deeper layers, use smaller learning rates.

**5. Compare and select best model**

Use validation metrics and failure analysis.

---

## 6. Common Mistakes to Avoid

| Mistake | Why it is problematic |
|---|---|
| Unfreezing everything immediately | Can cause unstable updates and overfitting |
| Using a large learning rate | Can damage useful pretrained weights |
| Training on very little data | Can cause poor generalisation |
| Ignoring BatchNorm behaviour | Running statistics may change unexpectedly |
| Comparing only training accuracy | Does not measure generalisation |
| Not saving the best checkpoint | May retain an inferior final model |

---

## 7. What You Should Remember 🧠

1. Fine-tuning adapts pretrained model weights to a new task.
2. Unfreezing allows selected layers to receive gradient updates.
3. Early CNN layers are often kept frozen because they learn relatively general features.
4. Deeper layers can be fine-tuned to learn task-specific representations.
5. Pretrained layers generally use smaller learning rates than newly created layers.
6. Catastrophic forgetting can occur when pretrained knowledge is disrupted by excessive updates.
7. Always compare fine-tuning against your feature extraction baseline.

**One-line understanding:**

> Feature extraction teaches a new classifier using existing features; fine-tuning also teaches selected parts of the backbone to improve those features.

---

## 8. Official References

- [PyTorch — Transfer Learning for Computer Vision](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial)
- [TensorFlow — Transfer Learning and Fine-Tuning](https://www.tensorflow.org/guide/keras/transfer_learning)
- [TensorFlow — Unfreezing and Fine-Tuning Pretrained Layers](https://www.tensorflow.org/tutorials/images/transfer_learning)
- [PyTorch — Official Training Implementation](https://github.com/pytorch/tutorials/blob/main/beginner_source/transfer_learning_tutorial.py)

---